In [ ]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import polars as pl
import polars_distance as pld
import shapely
import splink.comparison_library as cl
from geopy import distance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.blocking_analysis import (
    count_comparisons_from_blocking_rule,
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
    n_largest_blocks,
)
from splink.exploratory import completeness_chart, profile_columns

from ml_deduplication.dataset.features_engineering import preprocess_features_dataset
from ml_deduplication.dataset.splink_adapter import features_to_entities_df
from ml_deduplication.evaluation.metrics.cluster import generate_full_cluster_report
from ml_deduplication.evaluation.metrics.pairwise import pairwise_metrics_from_clusters
from ml_deduplication.inference.run_inference import query_acteurs
from ml_deduplication.training.utils import (
    create_acteur_to_cluster_dict,
    create_cluster_to_acteurs_dict,
    split_train_dev,
)

from sentence_transformers import SentenceTransformer

In [ ]:
DATABASE_URI = os.environ["DATABASE_CONNECTION_URI"]

In [ ]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

db_api = DuckDBAPI(connection=con)

In [ ]:
DATASETS_PATH = Path("../../datasets")
assert DATASETS_PATH.exists()

# Chargement des données


In [ ]:
df_features = pl.read_parquet(DATASETS_PATH / "features_dataset_20260818_4k.parquet")

In [ ]:
df_features.describe()

In [ ]:
df_features_splink_train, _ = features_to_entities_df(
    df_features.filter(pl.col("split") == "train")
)

In [ ]:
df_features_splink_train = df_features_splink_train.rename(
    {"cluster_id": "cluster_id_true"}
)

In [ ]:
df_features_splink_train

In [ ]:
df_features_splink_train.describe()

# Analyse feature Splink


In [ ]:
completeness_chart(df_features_splink_train, db_api=db_api)

In [ ]:
profile_columns(df_features_splink_train, db_api=db_api, top_n=10, bottom_n=5)

# Features


In [ ]:
def strip_ville_from_name(data: dict) -> str:
    if (data["ville"] is None) or (data["ville"].strip() == ""):
        return data["nom"]

    return data["nom"].replace(data["ville"].strip(), "")


def preprocess_features(
    df_features: pl.DataFrame, tfidf_vectorizer: TfidfVectorizer | None = None
) -> pl.DataFrame:
    df_features_preprocessed = df_features.clone()
    df_features_preprocessed = df_features_preprocessed.with_columns(
        pl.selectors.by_dtype(pl.String)
        .exclude(["cluster_id", "cluster_id_split", "label"])
        .str.strip_chars()
        .replace("__empty__", None)
        .replace("", None)
    )
    str_column_to_process = [
        "nom",
        "nom_commercial",
        "adresse",
        "adresse_complement",
        "ville",
    ]

    df_features_preprocessed = df_features_preprocessed.with_columns(
        pl.col(e)
        .str.to_lowercase()
        .str.normalize("NFKD")
        .map_elements(lambda x: x.encode("ASCII", "ignore").decode("utf-8"))
        for e in str_column_to_process
    ).with_columns(
        pl.selectors.starts_with("latitude").clip(-90.0, 90.0),
        pl.selectors.starts_with("longitude").clip(-180.0, 180.0),
        pl.struct(
            pl.concat_str(
                "nom", "nom_commercial", separator=" ", ignore_nulls=True
            ).alias("nom"),
            "ville",
        )
        .map_elements(strip_ville_from_name, return_dtype=pl.String)
        .alias(
            "nom_clean"
        ),  # Concatenate nom and nom_commercial and eliminate ville from nom
        pl.concat_str(
            pl.col("adresse").fill_null(""),
            pl.col("adresse_complement").fill_null(""),
            separator=" ",
        ).alias("adresse_clean"),  # Concatenate adresse and adresse_complement
        pl.coalesce(["nom", "nom_commercial"]).alias("nom"),
        pl.coalesce(["nom_commercial", "nom"]).alias("nom_commercial"),
        pl.concat_arr(["nom", "nom_commercial"]).alias("noms"),
    )

    addresses_texts = df_features_preprocessed.get_column("adresse_clean").to_list()

    if tfidf_vectorizer is None:
        tfidf_vectorizer = TfidfVectorizer(
            lowercase=False,
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=2,
            max_df=0.95,
        )
        corpus_train = df_features_preprocessed.get_column("adresse_clean").to_list()
        tfidf_vectorizer.fit(corpus_train)

    addresses_vectors = tfidf_vectorizer.transform(addresses_texts)

    df_features_preprocessed = df_features_preprocessed.with_columns(
        pl.lit(addresses_vectors.todense()).alias("adresse_clean_vector")
    ).with_columns(
        pl.when(pl.col("adresse").is_null() & pl.col("adresse_complement").is_null())
        .then(None)
        .otherwise("adresse_clean_vector")
        .alias("adresse_clean_vector")
    )

    return df_features_preprocessed, tfidf_vectorizer

## Adresse Tf-idf - Deprecated


In [ ]:
df_adresses = pl.read_csv(DATASETS_PATH / "adresses.csv")

In [ ]:
df_adresses = df_adresses.with_columns(
    pl.col("adresse")
    .str.to_lowercase()
    .str.normalize("NFKD")
    .map_elements(lambda x: x.encode("ASCII", "ignore").decode("utf-8"))
)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
corpus = df_adresses.get_column("adresse").to_list()
tfidf_vectorizer = tfidf_vectorizer.fit(corpus)
len(tfidf_vectorizer.vocabulary_)

In [ ]:
df_adresses.get_column("adresse").to_numpy()[:2]

In [ ]:
vectors = tfidf_vectorizer.transform(
    df_adresses.get_column("adresse").to_numpy()[:2]
).todense()

In [ ]:
vectors[0].tolist()[0]

In [ ]:
cosine_similarity(np.asarray(vectors))

In [ ]:
pl.Config.set_tbl_rows(30)
pl.DataFrame(
    {
        "vocab": dict(
            sorted(tfidf_vectorizer.vocabulary_.items(), key=lambda x: x[1])
        ).keys(),
        "vector_1": vectors[0].tolist()[0],
        "vector_2": vectors[1].tolist()[0],
    },
).filter((pl.col("vector_1") > 0) | (pl.col("vector_2") > 0))

## Embeddings


In [ ]:
model = SentenceTransformer("Lajavaness/sentence-camembert-large")

In [ ]:
model

# Blocking


In [ ]:
business_rules_fragment = "AND (((l.acteur_type_id = r.acteur_type_id) OR (l.acteur_type_id = 4 AND r.acteur_type_id = 3) OR (l.acteur_type_id = 3 AND r.acteur_type_id = 4)) AND (l.source_id!=r.source_id))"

In [ ]:
blocking_rules_for_analysis = [
    "(substr(l.code_postal,1,2) == substr(r.code_postal,1,2))"
    + business_rules_fragment,
    "(l.siren == r.siren)" + business_rules_fragment,
    "(ST_DISTANCE_SPHEROID(ST_Point(l.latitude,l.longitude),ST_Point(r.latitude,r.longitude)) < 30000 )"
    + business_rules_fragment,
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df_features_splink_train_preprocessed,
    blocking_rules=blocking_rules_for_analysis,
    db_api=db_api,
    link_type="dedupe_only",
    unique_id_column_name="entity_id",
)

# Linker


In [ ]:
business_rules_fragment = """AND 
    (
        (
            (l.acteur_type_id = r.acteur_type_id) 
            OR (l.acteur_type_id = 4 AND r.acteur_type_id = 3) 
            OR (l.acteur_type_id = 3 AND r.acteur_type_id = 4)
        ) 
        AND (l.source_id!=r.source_id)
    )"""

In [ ]:
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("nom"),
        cl.NameComparison("nom_commercial"),
        cl.ArrayIntersectAtSizes("noms", [1, 2]),
        cl.CosineSimilarityAtThresholds(
            "adresse_clean_vector", np.arange(0.95, 0.45, -0.05)
        ),
        cl.JaroWinklerAtThresholds("ville", [0.99, 0.98, 0.95, 0.90]),
        cl.EmailComparison("email"),
        cl.ExactMatch("siren"),
        cl.ExactMatch("siret"),
        cl.ExactMatch("telephone"),
        cl.ExactMatch("code_commune_insee"),
        cl.ExactMatch("naf_principal"),
        cl.CustomComparison(
            output_column_name="code_postal_custom",
            comparison_levels=[
                {
                    "sql_condition": '"code_postal_l" IS NULL OR "code_postal_r" IS NULL',
                    "label_for_charts": "code_postal is NULL",
                    "is_null_level": True,
                },
                {
                    "sql_condition": "code_postal_l == code_postal_r",
                    "label_for_charts": "code postal equals",
                },
                {
                    "sql_condition": "code_postal_l[1:2] == code_postal_r[1:2]",
                    "label_for_charts": "code postal department equals",
                },
                {"sql_condition": "ELSE", "label_for_charts": "All other comparisons"},
            ],
            comparison_description="ExactMatch",
        ),
        {
            "output_column_name": "location_custom",
            "comparison_levels": [
                {
                    "sql_condition": '"latitude_l" IS NULL OR "latitude_r" IS NULL OR "longitude_l" IS NULL OR "longitude_r" IS NULL',
                    "label_for_charts": "location is NULL",
                    "is_null_level": True,
                },
                *[
                    {
                        "sql_condition": f"ST_DISTANCE_SPHEROID(ST_Point(latitude_l,longitude_l),ST_Point(latitude_r,longitude_r)) < {e}",
                        "label_for_charts": f"location within {e}m",
                    }
                    for e in [5, 25, 50, 100, 500, 1000, 5000]
                ],
                {"sql_condition": "ELSE", "label_for_charts": "All other comparisons"},
            ],
            "comparison_description": "ExactMatch",
        },
    ],
    blocking_rules_to_generate_predictions=[
        "(substr(l.code_postal,1,2) == substr(r.code_postal,1,2))"
        + business_rules_fragment,
        "(l.siren == r.siren)" + business_rules_fragment,
        "(ST_DISTANCE_SPHEROID(ST_Point(l.latitude,l.longitude),ST_Point(r.latitude,r.longitude)) < 30000 )"
        + business_rules_fragment,
    ],
    retain_intermediate_calculation_columns=True,
    unique_id_column_name="entity_id",
    additional_columns_to_retain=[
        "cluster_id_true",
        "split",
    ],
)

## Training pipeline


In [ ]:
def create_ducbdb_backend(tmp_dir: Path = None) -> DuckDBAPI:
    con = duckdb.connect()
    con.install_extension("spatial")
    con.load_extension("spatial")

    if tmp_dir is not None:
        con.sql(f"SET temp_directory = '{tmp_dir.absolute()}' ")

    db_api = DuckDBAPI(connection=con)
    return db_api


def select_best_threshold(
    evaluation_data: list[dict], min_precision: float = 0.90
) -> dict:
    best_f_0_5 = 0
    best_f_0_5_index = 0

    best_threshold_weight = 0
    best_threshold_probability = 0
    for i, evaluation_dict in enumerate(evaluation_data):
        if evaluation_dict["precision"] >= min_precision:
            return evaluation_dict

        if evaluation_dict["f0_5"] > best_f_0_5:
            best_f_0_5_index = i

    print("Precision aimed not reached, defaulting to best f0_5")
    return evaluation_data[best_f_0_5_index]


def train_linker(linker: Linker) -> Linker:
    # Training
    deterministic_rules = [
        "(l.nom == r.nom)"
        "AND (l.adresse == r.adresse)"
        "AND (l.ville == r.ville)" + business_rules_fragment,
        "(l.siret == r.siret)"
        "AND (l.adresse == r.adresse)"
        "AND (l.ville == r.ville)" + business_rules_fragment,
        "(l.nom_commercial == r.nom_commercial)"
        "AND (l.adresse == r.adresse)"
        "AND (l.ville == r.ville)" + business_rules_fragment,
    ]

    linker.training.estimate_probability_two_random_records_match(
        deterministic_rules, recall=0.8
    )
    linker.training.estimate_u_using_random_sampling(max_pairs=1e9)
    linker.training.estimate_m_from_label_column("cluster_id_true")

    return linker


def create_linker(
    linker_settings: SettingsCreator,
    df_data_preprocessed: pl.DataFrame,
    db_api: DuckDBAPI,
) -> Linker:
    linker = Linker(
        df_data_preprocessed,
        linker_settings,
        db_api=db_api,
    )
    return linker


def training_pipeline(
    linker_settings: SettingsCreator,
    df_pairs_train: pl.DataFrame,
    min_precision: float = 0.95,
) -> tuple[Linker, dict]:

    # Split train/dev for threshold selection
    df_pairs_train_sub, df_pairs_dev = split_train_dev(df_pairs_train)

    # Turning train df_pairs  into long format
    df_train_sub, _ = features_to_entities_df(df_pairs_train_sub)
    df_train_sub = df_train_sub.rename({"cluster_id": "cluster_id_true"})
    # preprocessing df_train

    df_train_sub_preprocessed, tf_idf_train_sub = preprocess_features(df_train_sub)

    # Create training linker
    db_api_train_sub = create_ducbdb_backend(Path("/Volumes/PRO-G40"))
    linker_train_sub = create_linker(
        linker_settings, df_train_sub_preprocessed, db_api_train_sub
    )
    # Train linker
    linker_train_sub = train_linker(linker_train_sub)

    # Dev evaluation
    ## dev preprocessing
    df_dev, _ = features_to_entities_df(df_pairs_dev)
    df_dev = df_dev.rename({"cluster_id": "cluster_id_true"})
    df_dev_preprocessed, _ = preprocess_features(df_dev, tf_idf_train_sub)

    ## Create new linker object
    db_api_dev = create_ducbdb_backend(Path("/Volumes/PRO-G40"))
    linker_dev = create_linker(
        linker_train_sub.misc.save_model_to_json(), df_dev_preprocessed, db_api_dev
    )

    threshold_selection_chart = (
        linker_dev.evaluation.accuracy_analysis_from_labels_column(
            labels_column_name="cluster_id_true", add_metrics=["f1", "f0_5"]
        )
    )
    evaluation_data = threshold_selection_chart.data.values.to_dict()
    best_thresold_data = select_best_threshold(evaluation_data, min_precision)

    best_threshold_weight = best_thresold_data["truth_threshold"]
    best_threshold_prob = best_thresold_data["match_probability"]

    print("Best eval stats :")
    print(best_thresold_data)

    # Re-train on all data
    # Turning train df_pairs  into long format
    df_train, _ = features_to_entities_df(df_pairs_train)
    df_train = df_train.rename({"cluster_id": "cluster_id_true"})
    # preprocessing df_train

    df_train_preprocessed, tf_idf_train = preprocess_features(df_train)

    # Create training linker
    db_api_train = create_ducbdb_backend(Path("/Volumes/PRO-G40"))
    linker_train = create_linker(linker_settings, df_train_preprocessed, db_api_train)
    # Train linker
    linker_train = train_linker(linker_train)

    return linker_train, best_thresold_data, threshold_selection_chart, tf_idf_train

In [ ]:
linker_trained, threshold_selection_chart, threshold_selection_chart, tf_idf_train = (
    training_pipeline(
        settings,
        df_features.filter(pl.col("split") == "train"),
    )
)

In [ ]:
threshold_selection_chart

In [ ]:
BEST_THRESOLD = 32.8

## Paramètres retenus


In [ ]:
linker_trained._settings_obj.as_dict()

In [ ]:
linker_trained.visualisations.match_weights_chart()

In [ ]:
linker_trained.visualisations.m_u_parameters_chart()

# Unlinkable


In [ ]:
linker_trained.evaluation.unlinkables_chart()

# Evaluation test set


In [ ]:
cluster_to_acteur_dict_test = create_cluster_to_acteurs_dict(
    df_features.filter(pl.col("split") == "test")
)

In [ ]:
acteur_to_cluster_id_dict_test = create_acteur_to_cluster_dict(
    cluster_to_acteur_dict_test
)
acteur_to_cluster_id_dict_test

In [ ]:
df_true_test = (
    df_features.filter(pl.col("split") == "test")
    .rename(
        {"identifiant_unique_i": "entity_id_l", "identifiant_unique_j": "entity_id_r"}
    )
    .with_columns(
        pl.when(pl.col("label"))
        .then(pl.lit(1.0))
        .otherwise(pl.lit(0.0))
        .alias("clerical_match_score")
    )
)

In [ ]:
df_true_test.group_by("label").len()

## Features engineering


In [ ]:
df_features_splink_test, _ = features_to_entities_df(
    df_features.filter(pl.col("split") == "test")
)
df_features_splink_test_preprocessed, _ = preprocess_features(
    df_features_splink_test, tf_idf_train
)

In [ ]:
df_features_splink_test_preprocessed

## Test linker


In [ ]:
db_api_test = create_ducbdb_backend(Path("/Volumes/PRO-G40"))
linker_test = create_linker(
    linker_trained.misc.save_model_to_json(),
    df_features_splink_test_preprocessed.rename({"cluster_id": "cluster_id_true"}),
    db_api_test,
)

## Inference


In [ ]:
df_predictions_test = linker_test.inference.predict()

In [ ]:
df_predictions_test.as_pandas_dataframe()

In [ ]:
df_test_joined = (
    pl.DataFrame(df_predictions_test.as_pandas_dataframe())
    .join(
        df_true_test.select(["entity_id_l", "entity_id_r", "cluster_id", "label"]),
        on=["entity_id_l", "entity_id_r"],
        how="outer",
        suffix="_true",
    )
    .select(
        [
            "match_weight",
            "match_probability",
            "label",
            "entity_id_l",
            "entity_id_r",
            "entity_id_l_true",
            "entity_id_r_true",
            "nom_l",
            "nom_r",
            "split_l",
            "split_r",
        ]
    )
)

df_test_joined.filter(pl.col("label").is_null() | pl.col("label"))

In [ ]:
df_test_joined.filter(
    pl.col("entity_id_l").is_not_null() & pl.col("entity_id_l_true").is_not_null()
)

In [ ]:
clusters = linker_test.clustering.cluster_pairwise_predictions_at_threshold(
    df_predictions_test, threshold_match_weight=BEST_THRESOLD
)

In [ ]:
df_clusters_test = clusters.as_pandas_dataframe()

In [ ]:
df_clusters_test

## Analyse des résultats


In [ ]:
linker_test.visualisations.comparison_viewer_dashboard(
    df_predictions_test, "scv.html", overwrite=True
)

# You can view the scv.html file in your browser, or inline in a notbook as follows
from IPython.display import IFrame

IFrame(src="./scv.html", width="100%", height=1200)

## Splink Metrics


In [ ]:
df_true_test

In [ ]:
labels_table = linker_test.table_management.register_labels_table(df_true_test)

### Faux negatifs


In [ ]:
BEST_THRESOLD

In [ ]:
df_false_negatives = linker_test.evaluation.prediction_errors_from_labels_table(
    labels_table,
    include_false_negatives=True,
    include_false_positives=False,
    threshold_match_probability=(2**BEST_THRESOLD) / ((2**BEST_THRESOLD) + 1),
)
false_negatives = df_false_negatives.as_record_dict(limit=15)
linker_test.visualisations.waterfall_chart(false_negatives, remove_sensitive_data=True)

In [ ]:
false_negatives[2]

### Faux positifs


In [ ]:
# Note I've picked a threshold match probability of 0.01 here because otherwise
# in this simple example there are no false positives
df_false_positives = linker_test.evaluation.prediction_errors_from_labels_table(
    labels_table,
    include_false_negatives=False,
    include_false_positives=True,
    threshold_match_probability=(2**BEST_THRESOLD) / ((2**BEST_THRESOLD) + 1),
)
false_postives = df_false_positives.as_record_dict(limit=50)
linker_test.visualisations.waterfall_chart(false_postives, remove_sensitive_data=True)

In [ ]:
false_postives[14]

In [ ]:
df_features.filter(
    (pl.col("split") == "test")
    & (
        (pl.col("identifiant_unique_i") == "refashion_TLC-REFASHION-PAV-3440071")
        | (pl.col("identifiant_unique_j") == "refashion_TLC-REFASHION-PAV-3440071")
    )
)

### Unlikable chart


In [ ]:
linker_test.evaluation.unlinkables_chart()

## Pairwise metrics


In [ ]:
def splink_cluster_df_to_dict(
    splink_cluster_df: pd.DataFrame,
) -> dict:
    """
    Convertit la sortie du clustering Splink en dict {id: cluster_id}.
    les cluster_id sont générés automatiquement.
    Les entités que dedupe n'a rattachées à aucun cluster (cas limite selon
    versions) sont explicitement placées dans un cluster singleton.
    """
    id_to_cluster = {}
    for _, row in splink_cluster_df.iterrows():
        id_to_cluster[row["entity_id"]] = f"c_{row['cluster_id']}"

    return id_to_cluster

In [ ]:
id_to_cluster_test_pred = splink_cluster_df_to_dict(df_clusters_test)
id_to_cluster_test_pred

In [ ]:
pairwise_metrics_from_clusters(acteur_to_cluster_id_dict_test, id_to_cluster_test_pred)

## Metrics from cluster


In [ ]:
clusterwise_metrics = generate_full_cluster_report(
    acteur_to_cluster_id_dict_test, id_to_cluster_test_pred
)

In [ ]:
clusterwise_metrics

# Inference Commerces


In [ ]:
df_acteurs = query_acteurs(DATABASE_URI)

In [ ]:
df_acteurs.filter(pl.col("identifiant_unique") == "aliapur_060222126839")

## Feature engineering


In [ ]:
df_acteurs_preprocessed, _ = preprocess_features(df_acteurs, tf_idf_train)

In [ ]:
df_acteurs_preprocessed

In [ ]:
completeness_chart(df_acteurs_preprocessed, db_api=db_api)

In [ ]:
df_acteurs_preprocessed.select("nom", "nom_commercial")

## Inference


In [ ]:
db_api_commerces = create_ducbdb_backend(Path("/Volumes/PRO-G40"))
linker_commerces = create_linker(
    linker_trained.misc.save_model_to_json(),
    df_acteurs_preprocessed.rename({"identifiant_unique": "entity_id"}).with_columns(
        pl.lit(None).alias("cluster_id_true"), pl.lit(None).alias("split")
    ),
    db_api_commerces,
)

In [ ]:
df_predictions_commerce = linker_commerces.inference.predict(
    threshold_match_weight=BEST_THRESOLD
)

## Clustering


In [ ]:
clusters_commerce = (
    linker_commerces.clustering.cluster_pairwise_predictions_at_threshold(
        df_predictions_commerce, threshold_match_weight=BEST_THRESOLD
    )
)

In [ ]:
df_clusters_commerce = clusters_commerce.as_pandas_dataframe()

In [ ]:
df_clusters_commerce.columns

In [ ]:
df_clusters_commerce.head()

In [ ]:
df_clusters_commerce.shape

## Export


In [ ]:
clusters_commerces_dict = splink_cluster_df_to_dict(df_clusters_commerce)

In [ ]:
cluster_sizes = {}
for cluster_id in clusters_commerces_dict.values():
    cid = cluster_id
    cluster_sizes[cid] = cluster_sizes.get(cid, 0) + 1
multi_clusters = {k: v for k, v in cluster_sizes.items() if v > 1}

In [ ]:
len(multi_clusters)

In [ ]:
graph_metrics = linker_commerces.clustering.compute_graph_metrics(
    df_predictions_commerce,
    clusters_commerce,
    threshold_match_probability=(2**BEST_THRESOLD) / (1 + (2**BEST_THRESOLD)),
)

In [ ]:
node_metrics = graph_metrics.nodes.as_pandas_dataframe()
edge_metrics = graph_metrics.edges.as_pandas_dataframe()
cluster_metrics = graph_metrics.clusters.as_pandas_dataframe()

In [ ]:
cluster_metrics

In [ ]:
def splink_cluster_df_to_results_dict(
    splink_cluster_df: pd.DataFrame, cluster_metrics: pd.DataFrame
) -> dict:
    """
    Convertit la sortie de `deduper.partition()` en dict contenant le cluster id et le score de confiance.
    les cluster_id sont générés automatiquement.
    Utile pour logger les résultats.
    """
    id_to_cluster = {}
    for cluster_id, df in splink_cluster_df.groupby("cluster_id"):
        if len(df) <= 1:
            continue
        cluster_stats = cluster_metrics[cluster_metrics["cluster_id"] == cluster_id]
        children = []
        for _, row in df.iterrows():
            children.append(
                {
                    "acteur_id": row["entity_id"],
                    "cluster_density": cluster_stats["density"].item(),
                    "cluster_centralisation": cluster_stats[
                        "cluster_centralisation"
                    ].item(),
                    "num_acteurs": cluster_stats["n_nodes"].item(),
                }
            )
        id_to_cluster[f"c_{cluster_id}"] = children

    return id_to_cluster

In [ ]:
results_commerce = splink_cluster_df_to_results_dict(
    df_clusters_commerce, cluster_metrics
)

In [ ]:
len(results_commerce)

In [ ]:
df_results_commerce = (
    pl.DataFrame(cluster_metrics)
    .filter(pl.col("n_nodes") > 1)
    .join(
        pl.DataFrame(df_clusters_commerce), on="cluster_id", how="left", validate="1:m"
    )
    .select(
        pl.col("entity_id").alias("identifiant_unique"),
        pl.format("c_{}", "cluster_id").alias("cluster_id"),
        pl.col("n_nodes").alias("acteurs_count"),
        pl.col("density"),
        pl.col("cluster_centralisation"),
    )
    .sort(["density", "cluster_id"], descending=[True, False])
)

In [ ]:
df_results_commerce

In [ ]:
pl.DataFrame(cluster_metrics).sort(pl.col("n_nodes"), descending=True)[
    "cluster_id"
].first()

df_res


In [ ]:
df_results_commerce.write_csv("../../outputs/inference_cluters_splink_20260817.csv")

## Analyse Faux positifs


In [ ]:
df_predictions_commerce

In [ ]:
BEST_THRESOLD

In [ ]:
fp_to_asses = linker_commerces.misc.query_sql(
    "SELECT * from __splink__df_predict_ef3f17b5d"
    " WHERE entity_id_l='ecopae_ECOPAEPDR00117' AND entity_id_r='pyreo_587'",
    "splink_df",
)
linker_test.visualisations.waterfall_chart(
    fp_to_asses.as_record_dict(), remove_sensitive_data=True
)

In [ ]:
fp_to_asses.as_record_dict()

In [ ]:
df_acteurs_preprocessed.filter(
    pl.col("identifiant_unique").is_in(["corepile_88-DIS-0243", "ecologic_88-DINT-05"])
)

In [ ]:
vectors = (
    df_acteurs_preprocessed.filter(
        pl.col("identifiant_unique").is_in(
            ["ecologic_88-DINT-05", "corepile_88-DIS-0243"]
        )
    )
    .select("adresse_clean_vector")
    .to_series()
    .to_list()
)

In [ ]:
df_tfidf = pl.DataFrame(
    {
        "vocab": dict(
            sorted(tf_idf_train.vocabulary_.items(), key=lambda x: x[1])
        ).keys(),
        "vector_1": vectors[0],
        "vector_2": vectors[1],
    }
)

In [ ]:
df_tfidf.filter((pl.col("vector_1") > 0) | (pl.col("vector_2") > 0))

In [ ]:
vectors_manual = tf_idf_train.transform(
    ["21 place de l'eglise ", "4 place de l eglise", "place de l eglise"]
).todense()

In [ ]:
cosine_similarity(np.asarray(vectors_manual))

In [ ]:
sum(vectors[0])

In [ ]:
cosine_similarity([vectors[0], vectors[1]])

In [ ]:
df_predictions_commerce_pd = df_predictions_commerce.as_pandas_dataframe()